# 01 · Preprocessing

**Goal:** turn the three zipped splits into clean, typed tables, ready for feature engineering. No features are engineered here; driver and vehicle roll-ups come in `02_features`.

| Step | What happens |
|---|---|
| 1. Load & integrity | Read with a real CSV parser (`CALL_NOTE` contains commas), stack the splits with a `SPLIT` column, check keys |
| 2. `CALL_NOTE` | Removed from modeling data (train-only, mentions bind outcome) → saved separately for EDA |
| 3. Redundant columns | `SPLIT_FLAG` + 6 columns that are exact transforms of others are verified, then dropped |
| 4. Missing values | Numeric NaN kept (GBMs handle it) + `*_MISSING` flags; categorical NaN → explicit `"Missing"` level |
| 5. Types & values | ZIP zero-padded to 5-char string, dates parsed, Yes/No → 0/1, categoricals → `category`, make/model split, invalid values capped |

**Outputs** (`data/processed/`): `policies.parquet`, `drivers.parquet`, `vehicles.parquet`, `call_notes.parquet`.

**Kernel:** `Python (travelers-umc)`, the project `.venv` (see `requirements.txt`).

In [1]:
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

DATA, OUT = Path('../data'), Path('../data/processed')
OUT.mkdir(exist_ok=True)

## 1. Load & integrity checks
- Read each CSV directly from its zip. Validation labels (`policies_validate_target.csv`) are merged onto the validation features.
- All splits are stacked into one table per entity, and a `SPLIT` column (`train` / `valid` / `holdout`) is added to policies. Holdout has `CONVERT_IND = NaN`.
- Checks: `POLICY_ID` unique; every driver/vehicle row maps to a policy and every policy has ≥1 driver and ≥1 vehicle; no duplicate `(POLICY_ID, CAR_NO)`; `NUM_DRIVERS` equals the driver row count.

In [2]:
def read(split, name):
    with zipfile.ZipFile(DATA / f'2026-travelers-umc-{split}.zip') as z:
        return pd.read_csv(z.open(f'{split}/{name}.csv'))

pol = pd.concat({'train':   read('training', 'policies_train'),
                 'valid':   read('validation', 'policies_validate_features').merge(read('validation', 'policies_validate_target'), on='POLICY_ID', validate='1:1'),
                 'holdout': read('holdout', 'policies_holdout_features')}, names=['SPLIT']).reset_index(0).reset_index(drop=True)
drv = pd.concat([read('training', 'drivers_train'), read('validation', 'drivers_validate'), read('holdout', 'drivers_holdout')], ignore_index=True)
veh = pd.concat([read('training', 'vehicles_train'), read('validation', 'vehicles_validate'), read('holdout', 'vehicles_holdout')], ignore_index=True)

assert pol.POLICY_ID.is_unique and not veh.duplicated(['POLICY_ID', 'CAR_NO']).any()
assert set(drv.POLICY_ID) == set(veh.POLICY_ID) == set(pol.POLICY_ID)
assert pol.NUM_DRIVERS.eq(pol.POLICY_ID.map(drv.POLICY_ID.value_counts())).all()
print({k: v.shape for k, v in dict(pol=pol, drv=drv, veh=veh).items()})
pol.groupby('SPLIT').CONVERT_IND.agg(['size', 'count', 'mean'])

{'pol': (36871, 27), 'drv': (79615, 12), 'veh': (126922, 12)}


,size,count,mean
SPLIT,,,
holdout,7375,0,NaN
train,22122,22122,0.111744
valid,7374,7374,0.111744


**Finding: vehicle counts don't match.** `TOTAL_NUM_VEH` (and the owned/leased/loaned counts) on the policy table agree with the number of rows in the vehicle table for only ~20% of policies, which is about what chance would give. The two sources look independently generated. **Both are kept as-is.** The feature step will build counts from the vehicle table and test both versions; no reconciliation is attempted.

In [3]:
n_veh_rows = pol.POLICY_ID.map(veh.POLICY_ID.value_counts())
print(f'TOTAL_NUM_VEH == vehicle rows: {pol.TOTAL_NUM_VEH.eq(n_veh_rows).mean():.1%}')
print(f'owned+leased+loaned == TOTAL_NUM_VEH: {pol[["NUM_OWNED_VEH", "NUM_LEASED_VEH", "NUM_LOANED_VEH"]].sum(axis=1).eq(pol.TOTAL_NUM_VEH).mean():.1%}')

TOTAL_NUM_VEH == vehicle rows: 20.5%
owned+leased+loaned == TOTAL_NUM_VEH: 100.0%


## 2. `CALL_NOTE`: remove from modeling data
Free text such as *"Why quote: … Why not bound: …"*. It exists **only in training** (the column is absent from validation and holdout) and refers to the bind outcome, so it can't be a model input. It's saved to `call_notes.parquet` for EDA and the presentation (e.g. reasons quotes were lost).

In [4]:
pol[['POLICY_ID', 'CONVERT_IND', 'CALL_NOTE']].dropna(subset='CALL_NOTE').to_parquet(OUT / 'call_notes.parquet', index=False)
pol = pol.drop(columns='CALL_NOTE')

## 3. Drop redundant columns
`SPLIT_FLAG` is `"Train"` in every split, so it carries no information. Each derived column is **verified** to be an exact transform of its source before dropping (`MONTHLY_PREMIUM` matches to rounding, ≤ $0.004):

| Dropped | Equals |
|---|---|
| `MONTHLY_PREMIUM` | `QUOTED_AMT / 12` |
| `CREDIT_SCORE_SCALED` | `CREDIT_SCORE / 100` |
| `DRIVER_AGE_MONTHS` | `DRIVER_AGE × 12` |
| `SAFETY_SCORE_ADJ` | `SAFETY_RATING × 1.25` |
| `VEHICLE_AGE_MONTHS` | `VEHICLE_AGE × 12` |
| `DEPRECIATION_FACTOR` | `min(VEHICLE_AGE × 0.08, 0.9)` |

In [5]:
checks = [(pol, 'MONTHLY_PREMIUM', pol.QUOTED_AMT / 12),      (pol, 'CREDIT_SCORE_SCALED', pol.CREDIT_SCORE / 100),
          (drv, 'DRIVER_AGE_MONTHS', drv.DRIVER_AGE * 12),    (drv, 'SAFETY_SCORE_ADJ', drv.SAFETY_RATING * 1.25),
          (veh, 'VEHICLE_AGE_MONTHS', veh.VEHICLE_AGE * 12),  (veh, 'DEPRECIATION_FACTOR', np.minimum(veh.VEHICLE_AGE * .08, .9))]
for df, col, expected in checks:  # NaN must line up too, so nothing is lost by dropping
    assert np.allclose(df[col], expected, atol=0.005, equal_nan=True), col
assert pol.SPLIT_FLAG.nunique() == 1
pol = pol.drop(columns=['SPLIT_FLAG', 'MONTHLY_PREMIUM', 'CREDIT_SCORE_SCALED'])
drv = drv.drop(columns=['DRIVER_AGE_MONTHS', 'SAFETY_SCORE_ADJ'])
veh = veh.drop(columns=['VEHICLE_AGE_MONTHS', 'DEPRECIATION_FACTOR'])

## 4. Missing values
- **Numeric** NaN stays NaN, because gradient boosting handles it natively. Policy-level numeric columns with any NaN get a `<COL>_MISSING` 0/1 flag, since missing values may be informative: e.g. quotes with missing `QUOTED_AMT` convert at ~5% vs ~11% overall. Driver/vehicle numeric NaN (`SAFETY_RATING`, `HIGH_EDU_IND`, `VEHICLE_AGE`) is left for the aggregation step.
- **Categorical** NaN becomes an explicit `"Missing"` level in every table. E.g. ~10% of `PRIOR_CARRIER_GRP` is missing, which may mean "no prior insurance". `ZIP_CD` is excluded and stays NaN.
- Nothing is imputed with a mean or median.

In [6]:
print(pd.concat({n: d.isna().sum()[lambda s: s > 0] for n, d in dict(pol=pol, drv=drv, veh=veh).items()}).to_frame('n_missing'))

num_na = [c for c in pol.select_dtypes('number') if pol[c].isna().any() and c not in ('CONVERT_IND', 'ZIP_CD')]
pol[[f'{c}_MISSING' for c in num_na]] = pol[num_na].isna().astype('int8')
pol['ZIP_CD_MISSING'] = pol.ZIP_CD.isna().astype('int8')
for df in (pol, drv, veh):
    cat = [c for c in df.select_dtypes(['object', 'string']) if c not in ('POLICY_ID', 'SPLIT', 'QUOTE_DT')]
    df[cat] = df[cat].fillna('Missing')

                       n_missing
pol ZIP_CD                   352
    QUOTED_AMT                87
    PRIOR_CARRIER_GRP       3738
    CREDIT_SCORE             224
    COV_PKG_TYPE             593
    CAT_ZONE                 181
    CONVERT_IND             7375
drv LIVING_STATUS             30
    SAFETY_RATING             62
    HIGH_EDU_IND             401
veh COLOR                    794
    VEHICLE_AGE              391


## 5. Types & value cleanup
- `ZIP_CD`: float → 5-char string, restoring the leading zeros CT/NJ ZIPs lost (e.g. `7031.0` → `"07031"`). `ZIP3` is also kept for coarser geography and external joins.
- `QUOTE_DT` → datetime.
- `DISCOUNT_IND` (Yes/No), `HOME_POLICY_IND` (Y/N) and the 0/1 float indicators → `int8`. The indicator floats have no NaN except `HIGH_EDU_IND`, which becomes nullable `Int8`.
- `CAT_ZONE` → nullable `Int8`. It's ordinal 1–5, so it stays numeric.
- `COV_PKG_TYPE` → ordered category (Low < Medium < High). Other string columns (incl. `SPLIT`) → `category`.
- `MAKE_MODEL` split on `" : "` into `MAKE` (19 levels) and `MODEL`. The original is kept, since make+model is the key for external vehicle data.
- **Invalid values:**
  - `DRIVER_AGE` > 100 (46 rows, max 135) is capped at 100.
  - `SAFETY_RATING` < 0 (79 rows) is floored at 0. The scale otherwise runs 0–100.
  - `QUOTED_AMT` ($15 to $108k) is **kept as-is**: extremes are rare, conversion clearly falls as price rises, and trees are unaffected by monotone scaling.
- `ACCIDENT_FREE_YRS` often exceeds years of driving (synthetic quirk). It's left untouched; consider capping in feature engineering.

In [7]:
pol['ZIP_CD'] = pol.ZIP_CD.astype('Int64').astype('string').str.zfill(5)
pol['ZIP3'] = pol.ZIP_CD.str[:3]
pol['QUOTE_DT'] = pd.to_datetime(pol.QUOTE_DT)
pol['DISCOUNT_IND'] = pol.DISCOUNT_IND.eq('Yes').astype('int8')
pol['HOME_POLICY_IND'] = pol.HOME_POLICY_IND.eq('Y').astype('int8')
pol = pol.astype({'MULTI_QUOTE_IND': 'int8', 'CAT_ZONE': 'Int8', 'COV_PKG_TYPE': pd.CategoricalDtype(['Missing', 'Low', 'Medium', 'High'], ordered=True)})
drv = drv.astype({'HIGH_EDU_IND': 'Int8', 'TELEMATICS_OPT_IN': 'int8'})
veh = veh.astype({'ANTI_THEFT_IND': 'int8', 'GARAGE_IND': 'int8'})
veh[['MAKE', 'MODEL']] = veh.MAKE_MODEL.str.split(' : ', n=1, expand=True)

drv['DRIVER_AGE'] = drv.DRIVER_AGE.clip(upper=100)
drv['SAFETY_RATING'] = drv.SAFETY_RATING.clip(lower=0)

for df in (pol, drv, veh):  # remaining strings → category (POLICY_ID / ZIP stay strings)
    for c in df.select_dtypes(['object', 'string']).columns.difference(['POLICY_ID', 'ZIP_CD', 'ZIP3']):
        df[c] = df[c].astype('category')

## 6. Final checks & save
Row counts are unchanged from the raw data (36,871 policies / 79,615 drivers / 126,922 vehicles), and the only NaN left is in numeric columns that were deliberately kept (plus `CONVERT_IND` on holdout).

In [8]:
assert (len(pol), len(drv), len(veh)) == (36871, 79615, 126922)
assert pol.loc[pol.SPLIT != 'holdout', 'CONVERT_IND'].notna().all() and pol.loc[pol.SPLIT == 'holdout', 'CONVERT_IND'].isna().all()
for name, df in dict(policies=pol, drivers=drv, vehicles=veh).items():
    df.to_parquet(OUT / f'{name}.parquet', index=False)
    print(f'{name}: {df.shape}, NaN cols: {df.columns[df.isna().any()].tolist()}')
pol.head(3)

policies: (36871, 28), NaN cols: ['ZIP_CD', 'QUOTED_AMT', 'CREDIT_SCORE', 'CAT_ZONE', 'CONVERT_IND', 'ZIP3']


drivers: (79615, 10), NaN cols: ['SAFETY_RATING', 'HIGH_EDU_IND']
vehicles: (126922, 12), NaN cols: ['VEHICLE_AGE']


,SPLIT,QUOTE_DT,DISCOUNT_IND,HOME_POLICY_IND,ZIP_CD,STATE_CD,QUOTED_AMT,PRIOR_CARRIER_GRP,CREDIT_SCORE,COV_PKG_TYPE,...,PRIMARY_PARKING,QUOTE_CHANNEL,PRIOR_CLAIMS_CNT,MULTI_QUOTE_IND,DAYS_TO_FOLLOWUP,QUOTED_AMT_MISSING,CREDIT_SCORE_MISSING,CAT_ZONE_MISSING,ZIP_CD_MISSING,ZIP3
0,train,2018-09-19,0,0,34683,FL,9114.0,Carrier_7,518.0,Medium,...,unknown,aggregator,0,1,22,0,0,0,0,346
1,train,2016-06-19,1,0,30326,GA,15780.0,Missing,531.0,High,...,unknown,agent,2,0,7,0,0,0,0,303
2,train,2018-02-25,1,1,33020,FL,1904.0,Carrier_7,706.0,Low,...,unknown,online,0,1,1,0,0,0,0,330
